In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, HTML
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

In [5]:
ev2 =  pd.read_csv('../../Datasets/evaluacion2.csv')
ev2.head(3)

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_original_price,log_purchased_last_month,log_total_reviews
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,No Badge,Sponsored,1,0,0,0.0,PC Components,Media,5.484755,0.000000,5.883322
1,OWC 250GB Aura Pro 6G Flash SSD Upgrade for 20...,4.6,No Badge,Sponsored,1,0,0,0.0,Storage & Memory Cards,Media,3.783962,0.000000,5.624018
2,HP 67XL Black High-yield Ink Cartridge | Works...,4.6,Best Seller,Organic,0,1,0,0.0,"Office Supplies, Ink & Toner",Media,3.607941,10.819798,11.523598


In [4]:
df_word2vec =  pd.read_csv('ev2_Word2vec.csv')
df_word2vec.head()

,product_title,product_rating,is_best_seller,is_sponsored,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,product_category,product_segment,log_original_price,log_purchased_last_month,log_total_reviews,extracted_brand
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,No Badge,Sponsored,1,0,0,0.0,PC Components,Media,5.484755,0.000000,5.883322,Owc
1,OWC 250GB Aura Pro 6G Flash SSD Upgrade for 20...,4.6,No Badge,Sponsored,1,0,0,0.0,Storage & Memory Cards,Media,3.783962,0.000000,5.624018,Owc
2,HP 67XL Black High-yield Ink Cartridge | Works...,4.6,Best Seller,Organic,0,1,0,0.0,"Office Supplies, Ink & Toner",Media,3.607941,10.819798,11.523598,Tp-Link
3,HP 67 Black/Tri-color Ink Cartridges for HP Pr...,4.6,No Badge,Organic,0,0,0,0.0,"Office Supplies, Ink & Toner",Media,3.804215,10.819798,10.986868,Tp-Link
4,"Sony ZX Series Wired On-Ear Headphones, Black ...",4.5,Best Seller,Organic,0,0,0,0.0,"Audio, Sound & Recording Gear",Baja,2.355178,9.210440,11.598433,Sony


In [7]:
categorical_cols = ['is_best_seller', 'is_sponsored', 'product_category', 
                    'product_segment']

# Aplicar One-Hot Encoding a todas las variables categóricas
ev2_encoded = pd.get_dummies(df_word2vec, columns=categorical_cols + ['extracted_brand'], 
                            drop_first=True)

ev2_encoded

,product_title,product_rating,buy_box_availability,sustainability_tags,has_coupon,discount_percentage,log_original_price,log_purchased_last_month,log_total_reviews,is_best_seller_Best Seller,...,extracted_brand_Samsung,extracted_brand_Scotch,extracted_brand_Seagate,extracted_brand_Sharpie,extracted_brand_Sony,extracted_brand_Texas,extracted_brand_Tp-Link,extracted_brand_Unknown,extracted_brand_Vivo,extracted_brand_Western Digital
0,OWC 2.0TB Aura Pro X2 (GEN 4) SSD Complete Upg...,4.4,1,0,0,0.00,5.484755,0.000000,5.883322,False,...,False,False,False,False,False,False,False,False,False,False
1,OWC 250GB Aura Pro 6G Flash SSD Upgrade for 20...,4.6,1,0,0,0.00,3.783962,0.000000,5.624018,False,...,False,False,False,False,False,False,False,False,False,False
2,HP 67XL Black High-yield Ink Cartridge | Works...,4.6,0,1,0,0.00,3.607941,10.819798,11.523598,True,...,False,False,False,False,False,False,True,False,False,False
3,HP 67 Black/Tri-color Ink Cartridges for HP Pr...,4.6,0,0,0,0.00,3.804215,10.819798,10.986868,False,...,False,False,False,False,False,False,True,False,False,False
4,"Sony ZX Series Wired On-Ear Headphones, Black ...",4.5,0,0,0,0.00,2.355178,9.210440,11.598433,True,...,False,False,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7712,"VIVO Universal Wooden Laptop Treadmill Desk, A...",4.0,1,0,0,0.00,4.394326,5.303305,7.140453,False,...,False,False,False,False,False,False,False,False,True,False
7713,Neewer NP-FZ100 2400mAh Battery for Sony A7R V...,4.6,0,0,1,0.26,3.555062,5.707110,6.734592,False,...,False,False,False,False,True,False,False,False,False,False
7714,Monoprice XLR Male to 1/4-Inch TRS Male Cable ...,4.7,1,0,0,0.35,2.832036,6.216606,8.868273,False,...,False,False,False,False,False,False,False,False,False,False
7715,Dell Desktop Computer Package Compatible with ...,3.9,0,0,0,0.00,4.820282,4.615121,6.683361,False,...,False,False,False,False,False,False,False,False,False,False


In [8]:
ev2_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7717 entries, 0 to 7716
Data columns (total 69 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   product_title                                                   7717 non-null   object 
 1   product_rating                                                  7717 non-null   float64
 2   buy_box_availability                                            7717 non-null   int64  
 3   sustainability_tags                                             7717 non-null   int64  
 4   has_coupon                                                      7717 non-null   int64  
 5   discount_percentage                                             7717 non-null   float64
 6   log_original_price                                              7717 non-null   float64
 7   log_purchased_last_month                           

In [9]:
target = 'log_original_price'

In [ ]:
ev2_encoded.drop(columns=['product_title'])

In [8]:
X = ev2.drop(columns=[target])
y = ev2[target]
y_euro = np.expm1(y)

In [9]:
# Extract text features
cats = X.select_dtypes(exclude=np.number).columns.tolist()

# Convert to Pandas category
for col in cats:
   X[col] = X[col].astype('category')

In [10]:
X.dtypes

product_title               category
product_rating               float64
is_best_seller              category
is_sponsored                category
buy_box_availability           int64
sustainability_tags            int64
has_coupon                     int64
discount_percentage          float64
product_category            category
product_segment             category
log_purchased_last_month     float64
log_total_reviews            float64
dtype: object

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test:  {X_test.shape}")

Dimensiones de X_train: (6173, 12)
Dimensiones de X_test:  (1544, 12)


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
# Guardar los títulos también separados
titles_train = ev2.loc[X_train.index, 'product_title']
titles_test = ev2.loc[X_test.index, 'product_title']

# 2. APLICAR SBERT SOLO EN TRAIN
print("Generando embeddings para TRAIN...")
model = SentenceTransformer('all-mpnet-base-v2')

embeddings_train = model.encode(
    titles_train.tolist(),
    batch_size=64,
    show_progress_bar=True
)

# 3. AJUSTAR PCA SOLO EN TRAIN
n_components = min(30, len(X_train) - 1, embeddings_train.shape[1])
pca = PCA(n_components=n_components, random_state=42)
embeddings_train_reduced = pca.fit_transform(embeddings_train)  # fit_transform en TRAIN

# 4. APLICAR PCA EN TEST (solo transform, NO fit)
print("Generando embeddings para TEST...")
embeddings_test = model.encode(
    titles_test.tolist(),
    batch_size=64,
    show_progress_bar=True
)
embeddings_test_reduced = pca.transform(embeddings_test)  # solo transform en TEST

# 5. CREAR DATAFRAMES
train_sbert_df = pd.DataFrame(
    embeddings_train_reduced,
    columns=[f'title_sbert_{i}' for i in range(n_components)],
    index=X_train.index
)

test_sbert_df = pd.DataFrame(
    embeddings_test_reduced,
    columns=[f'title_sbert_{i}' for i in range(n_components)],
    index=X_test.index
)

# 6. COMBINAR CON FEATURES ORIGINALES
X_train_final = pd.concat([X_train, train_sbert_df], axis=1)
X_test_final = pd.concat([X_test, test_sbert_df], axis=1)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb

# 1. SPLIT PRIMERO
X_base = df[['product_rating', 'is_best_seller', 'is_sponsored', 
             'buy_box_availability', 'sustainability_tags', 'has_coupon',
             'discount_percentage', 'product_category', 'product_segment',
             'log_purchased_last_month', 'log_total_reviews']]
y = df['price']

X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=42
)

# 2. EXTRAER FEATURES MANUALES (en train y test por separado)
train_titles = df.loc[X_train_base.index, 'product_title']
test_titles = df.loc[X_test_base.index, 'product_title']

train_title_features = train_titles.apply(extract_amazon_title_features).apply(pd.Series)
test_title_features = test_titles.apply(extract_amazon_title_features).apply(pd.Series)

# 3. COMBINAR
X_train_final = pd.concat([
    X_train_base.reset_index(drop=True), 
    train_title_features.reset_index(drop=True)
], axis=1)

X_test_final = pd.concat([
    X_test_base.reset_index(drop=True), 
    test_title_features.reset_index(drop=True)
], axis=1)

print(f"Features totales: {X_train_final.shape[1]}")
print(f"Features de título añadidas: {len(train_title_features.columns)}")

# 4. ENTRENAR XGBOOST
dtrain = xgb.DMatrix(X_train_final, label=y_train, enable_categorical=True)
dtest = xgb.DMatrix(X_test_final, label=y_test, enable_categorical=True)

params = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'tree_method': 'hist',
    'max_depth': 9,
    'learning_rate': 0.05,  # Reducir un poco con más features
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 1,
    'reg_alpha': 1,
    'reg_lambda': 5,  # Más regularización
    'random_state': 42
}

evals = [(dtrain, 'train'), (dtest, 'test')]
model = xgb.train(
    params,
    dtrain,
    num_boost_round=3000,
    evals=evals,
    early_stopping_rounds=50,
    verbose_eval=100
)

# 5. EVALUAR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = model.predict(dtest)

print("\n" + "="*50)
print("📊 MÉTRICAS CON FEATURES MANUALES")
print("="*50)
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} €")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} €")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAPE: {np.mean(np.abs((y_test - y_pred) / y_test)) * 100:.2f}%")

# 6. VER QUÉ FEATURES DE TÍTULO SON MÁS IMPORTANTES
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
xgb.plot_importance(model, max_num_features=20, importance_type='gain', ax=ax)
plt.title('Top 20 Features Más Importantes')
plt.tight_layout()
plt.show()

# Filtrar solo features de título
gain_importance = model.get_booster().get_score(importance_type='gain')
title_feature_importance = {
    k: v for k, v in gain_importance.items() 
    if any(prefix in k for prefix in ['storage', 'ram', 'screen', 'pack', 'premium', 'brand', 'generation'])
}

print("\n📌 Top features de título:")
for feat, score in sorted(title_feature_importance.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"{feat}: {score:.2f}")